# LeetCode 3510 — Minimum Pair Removal to Sort Array II bằng Python

- Link: https://leetcode.com/problems/minimum-pair-removal-to-sort-array-ii/
- Độ khó: **Hard**
- Chủ đề: **Mô phỏng hiệu quả**: **heap xóa lười** (chọn cặp kề có tổng nhỏ nhất, lệch trái), **danh sách liên kết đôi** trên chỉ số, **đếm số "cặp xấu"** cập nhật cục bộ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumPairRemoval(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Nhận ra thao tác là **bắt buộc** (luôn gộp cặp kề có tổng nhỏ nhất, lệch trái nhất) ⇒ đáp án = số bước **mô phỏng** cho tới khi mảng không giảm. Thử thách là mô phỏng nhanh.
2. Dùng **heap** chứa `(tổng, chỉ số trái)` để lấy cặp cần gộp; tie-break "trái nhất" tự nhiên nhờ so sánh chỉ số.
3. Dùng **danh sách liên kết đôi** (`prv`, `nxt`) để xóa phần tử `O(1)` mà vẫn giữ chỉ số gốc.
4. Duy trì **số cặp kề giảm** (`bad`) — chỉ 3 cặp quanh vị trí gộp thay đổi ⇒ kiểm tra "đã sắp xếp chưa" trong `O(1)`.
5. Xử lý **phần tử lỗi thời** trong heap bằng kiểm tra lại khi lấy ra.

## 2. Đề bài & ví dụ

Cho mảng `nums`. Mỗi thao tác:

- Chọn cặp **kề nhau** có **tổng nhỏ nhất**; nếu nhiều cặp bằng nhau thì chọn cặp **bên trái nhất**.
- Thay cặp đó bằng **tổng** của chúng.

Trả về số thao tác **ít nhất** để mảng trở thành **không giảm**.

**Ví dụ 1:** `[5,2,3,1]` → `2`: gộp `(3,1)` → `[5,2,4]`; gộp `(2,4)` → `[5,6]`.

**Ví dụ 2:** `[1,2,2]` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `−10^9 ≤ nums[i] ≤ 10^9`.

In [1]:
from typing import List
import heapq
examples = [([5, 2, 3, 1], 2), ([1, 2, 2], 0)]
for a, exp in examples:
    print(a, '-> kỳ vọng', exp)

[5, 2, 3, 1] -> kỳ vọng 2
[1, 2, 2] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Mảng không giảm và "cặp xấu"

Mảng không giảm ⇔ không có chỉ số `i` với `a[i] > a[i+1]`. Gọi số cặp như vậy là `bad`; mảng đã sắp ⇔ `bad = 0`.

In [2]:
def count_bad(a):
    return sum(1 for i in range(len(a) - 1) if a[i] > a[i + 1])

print(count_bad([5, 2, 3, 1]), count_bad([1, 2, 2]))

2 0


### 3.2. Heap và so sánh bộ

`heapq` so sánh bộ `(tổng, chỉ số)` theo thứ tự từ điển: tổng nhỏ trước, cùng tổng thì chỉ số nhỏ (trái) trước — đúng quy tắc chọn của đề.

In [3]:
h = [(4, 2), (4, 0), (7, 1)]
heapq.heapify(h)
print(heapq.heappop(h))

(4, 0)


### 3.3. Danh sách liên kết đôi trên chỉ số

Giữ `prv[i]`, `nxt[i]`, `alive[i]`. Xóa `j` (đứng sau `i`): `nxt[i] = nxt[j]`, `prv[nxt[j]] = i`. Chỉ số gốc không đổi ⇒ thứ tự trái/phải giữ nguyên.

In [4]:
n = 5
nxt, prv = list(range(1, n + 1)), list(range(-1, n - 1))
def remove_after(i):
    j = nxt[i]
    nxt[i] = nxt[j]
    if nxt[j] < n:
        prv[nxt[j]] = i
remove_after(1)
i, seq = 0, []
while i < n:
    seq.append(i); i = nxt[i]
print('các chỉ số còn lại:', seq)

các chỉ số còn lại: [0, 1, 3, 4]


### 3.4. Heap xóa lười

Sau khi gộp, một số cặp trong heap không còn đúng (phần tử đã bị xóa hoặc giá trị đổi). Thay vì xóa khỏi heap, ta **kiểm tra khi lấy ra**: cặp `(s, i)` còn hợp lệ ⇔ `i` còn sống, `j = nxt[i]` tồn tại và `a[i] + a[j] == s`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Không có lựa chọn nào: mỗi bước đã được đề quy định. Vậy "ít nhất" = "số bước cho đến lần đầu mảng không giảm".

Mô phỏng ví dụ 1:

| bước | mảng | các tổng kề | gộp |
|---|---|---|---|
| 0 | `[5,2,3,1]` (bad = 2) | 7, 5, **4** | `(3,1)` |
| 1 | `[5,2,4]` (bad = 1) | 7, **6** | `(2,4)` |
| 2 | `[5,6]` (bad = 0) | | dừng |

Khi gộp cặp `(i, j)` (với `p` đứng trước `i`, `q` đứng sau `j`), chỉ các cặp `(p,i)`, `(i,j)`, `(j,q)` biến mất và `(p,i')`, `(i',q)` xuất hiện ⇒ cập nhật `bad` cục bộ.

## 5. Cách 1 — Brute Force: mô phỏng trực tiếp trên `list`

In [5]:
def pair_removal_brute(nums):
    a = nums[:]
    ops = 0
    while count_bad(a):
        i = min(range(len(a) - 1), key=lambda t: (a[t] + a[t + 1], t))
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1
    return ops

for a, exp in examples:
    print(pair_removal_brute(a), '| kỳ vọng', exp)

2 | kỳ vọng 2
0 | kỳ vọng 0


**Độ phức tạp:** tối đa `n − 1` bước, mỗi bước `O(n)` (tìm min, kiểm tra sắp xếp, xóa khỏi list) ⇒ `O(n^2)` = `10^{10}`.

**Nút thắt:** (1) tìm cặp tổng nhỏ nhất; (2) kiểm tra đã sắp xếp; (3) xóa phần tử giữa list.

## 6. Ý tưởng thuật toán / bước đột phá

Giải quyết từng nút thắt:

1. **Tìm min** ⇒ heap `(tổng, chỉ số trái)` + xóa lười.
2. **Kiểm tra sắp xếp** ⇒ duy trì `bad`; khi gộp `(i, j)` thành giá trị `s`:
   - trừ đóng góp cũ: `[a[p] > a[i]]`, `[a[i] > a[j]]`, `[a[j] > a[q]]`;
   - cộng đóng góp mới: `[a[p] > s]`, `[s > a[q]]`.
3. **Xóa giữa mảng** ⇒ danh sách liên kết đôi trên chỉ số gốc (giữ phần tử trái `i`, xóa `j`).

Sau gộp, đẩy hai cặp mới `(a[p] + s, p)` và `(s + a[q], i)` vào heap. Mỗi bước `O(log n)` ⇒ tổng `O(n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Mô phỏng `list` nhưng duy trì `bad` (vẫn `O(n^2)`)

Bỏ được việc kiểm tra sắp xếp `O(n)`, nhưng tìm min và xóa vẫn `O(n)`.

In [6]:
def pair_removal_bad_counter(nums):
    a = nums[:]
    bad = count_bad(a)
    ops = 0
    while bad:
        i = min(range(len(a) - 1), key=lambda t: (a[t] + a[t + 1], t))
        s = a[i] + a[i + 1]
        if i > 0: bad -= a[i - 1] > a[i]
        bad -= a[i] > a[i + 1]
        if i + 2 < len(a): bad -= a[i + 1] > a[i + 2]
        a[i:i + 2] = [s]
        if i > 0: bad += a[i - 1] > s
        if i + 1 < len(a): bad += s > a[i + 1]
        ops += 1
    return ops

for a, exp in examples:
    print(pair_removal_bad_counter(a), '| kỳ vọng', exp)

2 | kỳ vọng 2
0 | kỳ vọng 0


### 7.2. Cách 3 — Heap xóa lười + danh sách liên kết + `bad` (lời giải chính)

In [7]:
def pair_removal_heap(nums):
    n = len(nums)
    a = nums[:]
    nxt = list(range(1, n + 1))
    prv = list(range(-1, n - 1))
    alive = [True] * n
    bad = count_bad(a)
    h = [(a[i] + a[i + 1], i) for i in range(n - 1)]
    heapq.heapify(h)
    ops = 0
    while bad:
        s, i = heapq.heappop(h)
        if not alive[i]:
            continue
        j = nxt[i]
        if j >= n or a[i] + a[j] != s:
            continue                       # phần tử lỗi thời
        p, q = prv[i], nxt[j]
        if p >= 0: bad -= a[p] > a[i]
        bad -= a[i] > a[j]
        if q < n: bad -= a[j] > a[q]
        a[i] = s
        alive[j] = False
        nxt[i] = q
        if q < n:
            prv[q] = i
        if p >= 0:
            bad += a[p] > s
            heapq.heappush(h, (a[p] + s, p))
        if q < n:
            bad += s > a[q]
            heapq.heappush(h, (s + a[q], i))
        ops += 1
    return ops

for a, exp in examples:
    print(pair_removal_heap(a), '| kỳ vọng', exp)

2 | kỳ vọng 2
0 | kỳ vọng 0


**Tại sao kiểm tra `a[i] + a[j] == s` là đủ?** Nếu phần tử trong heap khớp với **trạng thái hiện tại** (đúng cặp kề đang tồn tại và đúng tổng), thì dù nó được đẩy vào từ lúc nào, nó cũng mô tả một cặp thật với tổng thật ⇒ an toàn để dùng.

**Độ phức tạp:** mỗi lần gộp đẩy ≤ 2 phần tử ⇒ heap có `O(n)` phần tử ⇒ `O(n log n)`.

### 7.3. Cách 4 — Cây phân đoạn lưu "tổng cặp" theo chỉ số trái

Thay heap bằng cây phân đoạn trên chỉ số gốc: lá `i` lưu `a[i] + a[nxt[i]]` nếu `i` còn sống và có phần tử sau, ngược lại `+∞`. Truy vấn min toàn cục (lệch trái khi bằng nhau), cập nhật điểm sau mỗi lần gộp. Không cần xóa lười.

In [8]:
def pair_removal_segtree(nums):
    n = len(nums)
    if n <= 1:
        return 0
    a = nums[:]
    nxt = list(range(1, n + 1))
    prv = list(range(-1, n - 1))
    INF = float('inf')
    size = 1
    while size < n:
        size *= 2
    t = [(INF, 0)] * (2 * size)
    for i in range(n - 1):
        t[size + i] = (a[i] + a[i + 1], i)
    for i in range(size - 1, 0, -1):
        t[i] = min(t[2 * i], t[2 * i + 1])

    def setv(i, v):
        i += size
        t[i] = (v, i - size)
        i >>= 1
        while i:
            t[i] = min(t[2 * i], t[2 * i + 1])
            i >>= 1

    bad = count_bad(a)
    ops = 0
    while bad:
        s, i = t[1]
        j = nxt[i]
        p, q = prv[i], nxt[j]
        if p >= 0: bad -= a[p] > a[i]
        bad -= a[i] > a[j]
        if q < n: bad -= a[j] > a[q]
        a[i] = s
        nxt[i] = q
        if q < n:
            prv[q] = i
        setv(j, INF)
        setv(i, s + a[q] if q < n else INF)
        if p >= 0:
            setv(p, a[p] + s)
            bad += a[p] > s
        if q < n:
            bad += s > a[q]
        ops += 1
    return ops

for a, exp in examples:
    print(pair_removal_segtree(a), '| kỳ vọng', exp)

2 | kỳ vọng 2
0 | kỳ vọng 0


Bộ `(tổng, chỉ số)` làm `min` tự chọn chỉ số nhỏ nhất khi tổng bằng nhau. Lá `INF` có chỉ số tùy ý nhưng không bao giờ được chọn khi `bad > 0` (luôn còn ít nhất một cặp).

## 8. TRACE từng bước

Trace cách 3 với `nums = [3, -1, 4, 1, -5, 9]`: in mảng hiện tại (theo danh sách liên kết), cặp được gộp, `bad`.

In [9]:
def trace(nums):
    n = len(nums)
    a = nums[:]
    nxt = list(range(1, n + 1)); prv = list(range(-1, n - 1)); alive = [True] * n
    bad = count_bad(a)
    h = [(a[i] + a[i + 1], i) for i in range(n - 1)]
    heapq.heapify(h)
    def current():
        i, out = 0, []
        while i < n:
            out.append(a[i]); i = nxt[i]
        return out
    print('ban đầu:', current(), '| bad =', bad)
    ops = 0
    while bad:
        s, i = heapq.heappop(h)
        if not alive[i] or nxt[i] >= n or a[i] + a[nxt[i]] != s:
            print(f"   bỏ phần tử lỗi thời ({s}, {i})")
            continue
        j = nxt[i]; p, q = prv[i], nxt[j]
        if p >= 0: bad -= a[p] > a[i]
        bad -= a[i] > a[j]
        if q < n: bad -= a[j] > a[q]
        old = (a[i], a[j])
        a[i] = s; alive[j] = False; nxt[i] = q
        if q < n: prv[q] = i
        if p >= 0: bad += a[p] > s; heapq.heappush(h, (a[p] + s, p))
        if q < n: bad += s > a[q]; heapq.heappush(h, (s + a[q], i))
        ops += 1
        print(f"bước {ops}: gộp {old} -> {s} | mảng {current()} | bad = {bad}")
    print('đáp án =', ops)

trace([3, -1, 4, 1, -5, 9])
print('vét cạn:', pair_removal_brute([3, -1, 4, 1, -5, 9]))

ban đầu: [3, -1, 4, 1, -5, 9] | bad = 3
bước 1: gộp (1, -5) -> -4 | mảng [3, -1, 4, -4, 9] | bad = 2
bước 2: gộp (4, -4) -> 0 | mảng [3, -1, 0, 9] | bad = 1
bước 3: gộp (-1, 0) -> -1 | mảng [3, -1, 9] | bad = 1
bước 4: gộp (3, -1) -> 2 | mảng [2, 9] | bad = 0
đáp án = 4
vét cạn: 4


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minimumPairRemoval(self, nums: List[int]) -> int:
        n = len(nums)
        a = nums[:]
        nxt = list(range(1, n + 1))
        prv = list(range(-1, n - 1))
        alive = [True] * n
        bad = sum(1 for i in range(n - 1) if a[i] > a[i + 1])
        h = [(a[i] + a[i + 1], i) for i in range(n - 1)]
        heapq.heapify(h)
        ops = 0
        while bad:
            s, i = heapq.heappop(h)
            if not alive[i]:
                continue
            j = nxt[i]
            if j >= n or a[i] + a[j] != s:
                continue
            p, q = prv[i], nxt[j]
            if p >= 0:
                bad -= a[p] > a[i]
            bad -= a[i] > a[j]
            if q < n:
                bad -= a[j] > a[q]
            a[i] = s
            alive[j] = False
            nxt[i] = q
            if q < n:
                prv[q] = i
                bad += s > a[q]
                heapq.heappush(h, (s + a[q], i))
            if p >= 0:
                bad += a[p] > s
                heapq.heappush(h, (a[p] + s, p))
            ops += 1
        return ops

for a, exp in examples:
    got = Solution().minimumPairRemoval(a)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
0 OK


## 10. Vì sao thuật toán đúng

1. **Quá trình tất định:** mỗi bước được quy định duy nhất (tổng nhỏ nhất, lệch trái nhất), nên số bước tối thiểu là số bước tới lần đầu mảng không giảm — mô phỏng đúng quy tắc là đủ.
2. **Danh sách liên kết giữ thứ tự:** gộp `(i, j)` thành phần tử tại `i` (bên trái) và xóa `j` ⇒ thứ tự các chỉ số gốc còn sống đúng là thứ tự trong mảng hiện tại. Vì vậy "trái nhất" ⇔ "chỉ số gốc nhỏ nhất".
3. **Heap đúng:** mọi cặp kề hiện tại có ít nhất một phần tử khớp trạng thái trong heap (được đẩy khi cặp hình thành). Phần tử khớp trạng thái là cặp thật; phần tử không khớp bị bỏ. Do đó phần tử khớp đầu tiên lấy ra là cặp có `(tổng, chỉ số)` nhỏ nhất.
4. **`bad` đúng:** chỉ các cặp kề có chứa `i` hoặc `j` thay đổi; ta trừ đóng góp cũ và cộng đóng góp mới của đúng các cặp đó.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng list | `O(n^2)` | `O(n)` |
| 2. List + `bad` | `O(n^2)` | `O(n)` |
| 3. Heap lười + liên kết | `O(n log n)` | `O(n)` |
| 4. Cây phân đoạn | `O(n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'bad_counter': pair_removal_bad_counter, 'heap': pair_removal_heap, 'segtree': pair_removal_segtree,
         'submit': lambda a: Solution().minimumPairRemoval(a)}
edge = [([7], 0), ([2, 1], 1), ([0, 0, 0], 0), ([-1, -2, -3], None), ([10**9, -10**9, 10**9, -10**9], None), ([3, 3, 1, 1], None)]
for a, exp in examples + edge:
    b = pair_removal_brute(a)
    if exp is not None:
        assert b == exp
    for name, f in impls.items():
        assert f(a) == b, (name, a)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với mô phỏng trực tiếp (nhiều tổng bằng nhau)

In [12]:
import random
random.seed(3510)
cnt = 0
for _ in range(2000):
    n = random.randint(1, 12)
    a = [random.randint(-4, 4) for _ in range(n)]
    exp = pair_removal_brute(a)
    for name, f in impls.items():
        assert f(a) == exp, (name, a)
    cnt += 1
print('Khớp mô phỏng trên', cnt, 'mảng ngẫu nhiên.')

Khớp mô phỏng trên 2000 mảng ngẫu nhiên.


### 12.3. Ca lớn (`n = 10^5`)

In [13]:
import time
random.seed(5)
n = 10**5
cases = {
    'ngẫu nhiên': [random.randint(-10**9, 10**9) for _ in range(n)],
    'giảm dần': list(range(n, 0, -1)),
    'xen kẽ lớn/nhỏ': [(-1) ** i * 10**9 for i in range(n)],
    'gần như tăng': list(range(n - 1)) + [-10**9],
}
for name, a in cases.items():
    t = time.time(); x = Solution().minimumPairRemoval(a); t1 = time.time() - t
    t = time.time(); y = pair_removal_segtree(a); t2 = time.time() - t
    assert x == y
    print(f"{name:>15}: {x} bước | heap {t1:.2f}s | cây phân đoạn {t2:.2f}s")

     ngẫu nhiên: 99999 bước | heap 0.22s | cây phân đoạn 0.93s


       giảm dần: 99999 bước | heap 0.43s | cây phân đoạn 1.00s


 xen kẽ lớn/nhỏ: 99998 bước | heap 0.20s | cây phân đoạn 0.70s


   gần như tăng: 99999 bước | heap 0.39s | cây phân đoạn 0.88s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam "gộp ở chỗ bị giảm" thay vì theo đúng quy tắc của đề

In [14]:
def wrong_fix_violation(nums):
    a, ops = nums[:], 0
    while count_bad(a):
        i = next(t for t in range(len(a) - 1) if a[t] > a[t + 1])   # SAI: không phải cặp tổng nhỏ nhất
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1
    return ops

random.seed(1)
found = next(a for a in ([random.randint(-5, 5) for _ in range(random.randint(2, 7))] for _ in range(5000))
             if wrong_fix_violation(a) != pair_removal_brute(a))
print('phản ví dụ:', found, '| sai:', wrong_fix_violation(found), '| đúng:', pair_removal_brute(found))

phản ví dụ: [5, 1, -2, -4, 2] | sai: 3 | đúng: 4


**Vì sao sai:** đề không cho chọn cặp; luôn phải gộp cặp **tổng nhỏ nhất**. Đây là bài mô phỏng, không phải tối ưu tự do.

### Lỗi 2 — Tie-break chọn cặp **phải nhất** (hoặc tùy ý)

In [15]:
def wrong_rightmost(nums):
    a, ops = nums[:], 0
    while count_bad(a):
        i = min(range(len(a) - 1), key=lambda t: (a[t] + a[t + 1], -t))     # SAI
        a[i:i + 2] = [a[i] + a[i + 1]]
        ops += 1
    return ops

random.seed(2)
found = next(a for a in ([random.randint(-3, 3) for _ in range(random.randint(2, 8))] for _ in range(5000))
             if wrong_rightmost(a) != pair_removal_brute(a))
print('phản ví dụ:', found, '| sai:', wrong_rightmost(found), '| đúng:', pair_removal_brute(found))

phản ví dụ: [3, -3, -3, -3, -1, 3, -2, 2] | sai: 7 | đúng: 5


**Vì sao sai:** khi nhiều cặp cùng tổng, chọn khác đi sẽ dẫn tới chuỗi trạng thái khác.

### Lỗi 3 — Không kiểm tra phần tử lỗi thời trong heap

In [16]:
def wrong_no_stale_check(nums):
    n = len(nums); a = nums[:]
    nxt = list(range(1, n + 1)); prv = list(range(-1, n - 1)); alive = [True] * n
    bad = count_bad(a)
    h = [(a[i] + a[i + 1], i) for i in range(n - 1)]; heapq.heapify(h)
    ops = 0
    while bad and h:
        s, i = heapq.heappop(h)
        if not alive[i] or nxt[i] >= n:
            continue                       # SAI: thiếu kiểm tra a[i] + a[j] == s
        j = nxt[i]; p, q = prv[i], nxt[j]
        if p >= 0: bad -= a[p] > a[i]
        bad -= a[i] > a[j]
        if q < n: bad -= a[j] > a[q]
        s = a[i] + a[j]
        a[i] = s; alive[j] = False; nxt[i] = q
        if q < n: prv[q] = i; bad += s > a[q]; heapq.heappush(h, (s + a[q], i))
        if p >= 0: bad += a[p] > s; heapq.heappush(h, (a[p] + s, p))
        ops += 1
    return ops

random.seed(3)
found = next(a for a in ([random.randint(-5, 5) for _ in range(random.randint(3, 9))] for _ in range(5000))
             if wrong_no_stale_check(a) != pair_removal_brute(a))
print('phản ví dụ:', found, '| sai:', wrong_no_stale_check(found), '| đúng:', pair_removal_brute(found))

phản ví dụ: [5, -1, 1, 3, 1, 4] | sai: 5 | đúng: 4


**Vì sao sai:** một phần tử `(s, i)` cũ có thể vẫn trỏ tới `i` còn sống nhưng tổng đã đổi ⇒ gộp sai cặp (không phải cặp nhỏ nhất hiện tại).

### Lỗi 4 — Dùng `list.pop(i)` cho mỗi lần gộp (vẫn `O(n^2)`)

In [17]:
import time
a = list(range(30000, 0, -1))
t = time.time(); pair_removal_bad_counter(a[:3000]); t_small = time.time() - t
t = time.time(); pair_removal_heap(a); t_heap = time.time() - t
print(f"list (n = 3000): {t_small:.2f}s | heap (n = 30000): {t_heap:.2f}s — list sẽ chậm ~100 lần khi n tăng 10 lần")

list (n = 3000): 0.45s | heap (n = 30000): 0.10s — list sẽ chậm ~100 lần khi n tăng 10 lần


**Vì sao sai:** xóa giữa list dịch chuyển phần còn lại; cộng với tìm min tuyến tính ⇒ `O(n^2)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Quy trình được quy định, đếm số bước" ⇒ **mô phỏng**; tối ưu cấu trúc dữ liệu cho từng thao tác.
- "Liên tục lấy min rồi gộp phần tử kề" ⇒ **heap + danh sách liên kết đôi** (giống thuật toán Huffman trên dãy, LC 2263, 1167).
- Điều kiện toàn cục (đã sắp xếp chưa) ⇒ đếm "vi phạm cục bộ" và cập nhật quanh vị trí thay đổi.
- Heap không xóa được giữa chừng ⇒ **xóa lười**: kiểm tra khi lấy ra.

## 15. Biến thể của bài

1. Bản I (LC 3507): `n ≤ 50` ⇒ mô phỏng trực tiếp là đủ.
2. Gộp cặp có **tích** nhỏ nhất / **hiệu** nhỏ nhất ⇒ cùng khung, đổi khóa heap.
3. Hỏi tổng **chi phí** (tổng các tổng được tạo) thay vì số bước ⇒ cộng `s` mỗi lần.
4. Bài liên quan: LC 1167 *Minimum Cost to Connect Sticks*, LC 2263, "Huffman trên dãy kề".

In [18]:
# Biến thể 3: tổng chi phí các lần gộp cho tới khi không giảm
def total_cost(nums):
    a, cost = nums[:], 0
    while count_bad(a):
        i = min(range(len(a) - 1), key=lambda t: (a[t] + a[t + 1], t))
        cost += a[i] + a[i + 1]
        a[i:i + 2] = [a[i] + a[i + 1]]
    return cost

print(total_cost([5, 2, 3, 1]))   # 4 + 6

10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Mô phỏng list | `O(n^2)` | `n ≤ 2000`, oracle |
| 2 | List + `bad` | `O(n^2)` | Hiểu cập nhật cục bộ |
| 3 | Heap lười + liên kết | `O(n log n)` | **Nộp bài** |
| 4 | Cây phân đoạn | `O(n log n)` | Không muốn xóa lười |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minimumPairRemoval([4, 3, 2, 1])`.

**Bài 2 (tính tay):** `minimumPairRemoval([1, 1, -1, 1])`.

**Bài 3 (điền TODO):** viết `update_bad(a, p, i, j, q, s)` trả về lượng thay đổi của `bad`.

**Bài 4 (tự viết lại):** biến thể 3 bằng heap (`O(n log n)`).

**Bài 5 (mini challenge):** trả về **mảng cuối cùng** (sau khi dừng) bằng cách 3.

In [19]:
def update_bad_todo(a, p, i, j, q, s, n):
    # TODO
    raise NotImplementedError

def total_cost_heap_todo(nums):
    # TODO
    raise NotImplementedError

def final_array_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tổng kề: 7, 5, **3** ⇒ gộp `(2,1)` → `[4,3,3]`. (2) Tiếp: 7, **6** ⇒ `[4,6]`. (3) `2`.

**Bài 2.** (1) Tổng kề: 2, **0**, 0 ⇒ gộp trái nhất `(1,−1)` → `[1,0,1]`. (2) Tổng 1, 1 ⇒ gộp `(1,0)` → `[1,1]`. (3) `2`.

**Bài 3.** (1) Trừ ba so sánh cũ. (2) Cộng hai so sánh mới. (3) Chú ý `p < 0` hoặc `q ≥ n`.

**Bài 4.** (1) Như lời giải chính. (2) Cộng `s` mỗi lần gộp. (3) So với `total_cost`.

**Bài 5.** (1) Sau vòng lặp, đi theo `nxt` từ chỉ số 0. (2) Lấy `a[i]`. (3) So với mô phỏng list.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
a1 = Solution().minimumPairRemoval([4, 3, 2, 1])
a2 = Solution().minimumPairRemoval([1, 1, -1, 1])
assert a1 == 2 == pair_removal_brute([4, 3, 2, 1]) and a2 == 2 == pair_removal_brute([1, 1, -1, 1])
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 2 | Bài 2: 2


In [21]:
# Bài 3, 4, 5
def update_bad(a, p, i, j, q, s, n):
    d = 0
    if p >= 0: d -= a[p] > a[i]
    d -= a[i] > a[j]
    if q < n: d -= a[j] > a[q]
    if p >= 0: d += a[p] > s
    if q < n: d += s > a[q]
    return d

def simulate(nums):
    n = len(nums); a = nums[:]
    nxt = list(range(1, n + 1)); prv = list(range(-1, n - 1)); alive = [True] * n
    bad = count_bad(a)
    h = [(a[i] + a[i + 1], i) for i in range(n - 1)]; heapq.heapify(h)
    cost = 0
    while bad:
        s, i = heapq.heappop(h)
        if not alive[i] or nxt[i] >= n or a[i] + a[nxt[i]] != s:
            continue
        j = nxt[i]; p, q = prv[i], nxt[j]
        bad += update_bad(a, p, i, j, q, s, n)
        cost += s
        a[i] = s; alive[j] = False; nxt[i] = q
        if q < n: prv[q] = i; heapq.heappush(h, (s + a[q], i))
        if p >= 0: heapq.heappush(h, (a[p] + s, p))
    out, i = [], 0
    while i < n:
        out.append(a[i]); i = nxt[i]
    return cost, out

def final_brute(nums):
    a = nums[:]
    while count_bad(a):
        i = min(range(len(a) - 1), key=lambda t: (a[t] + a[t + 1], t))
        a[i:i + 2] = [a[i] + a[i + 1]]
    return a

random.seed(4)
for _ in range(500):
    a = [random.randint(-5, 5) for _ in range(random.randint(1, 10))]
    cost, out = simulate(a)
    assert cost == total_cost(a) and out == final_brute(a)
print('Bài 3-5: ví dụ', simulate([5, 2, 3, 1]))

Bài 3-5: ví dụ (10, [5, 6])


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nxt, prv, alive trên chỉ số gốc; bad = số i có a[i] > a[i+1]
heap = [(a[i] + a[i+1], i)]
while bad:
    (s, i) = pop; bỏ nếu i chết / không có j = nxt[i] / a[i] + a[j] != s
    cập nhật bad cho (p,i), (i,j), (j,q) -> (p,i'), (i',q)
    a[i] = s; xóa j; push (a[p]+s, p), (s+a[q], i); ops += 1
```

**Checklist:**

- [ ] Gộp đúng cặp **tổng nhỏ nhất, trái nhất** (không tự chọn)?
- [ ] Kiểm tra phần tử lỗi thời bằng **tổng hiện tại**?
- [ ] Giữ phần tử **trái** khi gộp để giữ thứ tự chỉ số?
- [ ] Cập nhật `bad` cho đủ 3 cặp cũ và 2 cặp mới?
- [ ] Mảng 1 phần tử / đã sắp xếp ⇒ 0?